# Phase 3 — Physics features (the 5-channel input)

**Why (paper §3.5):** instead of hoping the network discovers the optics of haze from
raw pixels, we hand it two pre-computed maps that come from physics and behave the same
in every city:

- **Transmission** (Dark Channel Prior): how much of the scene's light survived the trip
  to the camera. **Low where haze is dense.** Clear photos always have *something* dark in
  each small patch; haze lifts those dark pixels.
- **Inverted saturation**: high where colour is washed out. Works on **sky**, exactly
  where the Dark Channel Prior fails (sky has nothing dark).

The model input becomes a **5-channel image**: Red, Green, Blue, transmission, inverted
saturation. We compute the two maps **once and cache them** — recomputing every epoch
would be far too slow.

## Bootstrap — run this first

This one cell makes the notebook self-contained: it grabs the code from GitHub (if it
isn't already here), installs the libraries, connects Google Drive, and makes our `src`
modules importable. **Set `REPO_URL` to your repository's URL.** It's safe to re-run and
also works on a laptop.

In [ ]:
# === Bootstrap — RUN ME FIRST (set REPO_URL to your repo) ===
REPO_URL = "https://github.com/YOUR_USERNAME/pm25-visual-aq.git"   # <-- EDIT THIS

import os, sys, subprocess

def _find_repo_root():
    # Are we already inside the repo (or just above the notebooks/ folder)?
    for cand in (".", "..", "pm25-visual-aq"):
        if os.path.isdir(os.path.join(cand, "src")):
            return os.path.abspath(cand)
    return None

_root = _find_repo_root()
if _root is None:                       # fresh session: clone the code
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "pm25-visual-aq"], check=True)
    _root = os.path.abspath("pm25-visual-aq")
else:                                   # reused clone (e.g. a stale Kaggle dir): pull the latest
    subprocess.run(["git", "-C", _root, "pull", "--ff-only"], check=False)
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=False)
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")

print("repo root:", _root, "| Colab:", IN_COLAB)

In [ ]:
from src.config import load_config
from src import data, physics
import numpy as np, matplotlib.pyplot as plt, os
cfg = load_config()

SOURCE = cfg["data"]["drive_path"]        # laptop test: "tests/fixture_ds"
ds, df = data.load_clean(SOURCE, from_disk=True, seed=cfg["seed"])
print("rows:", len(df))

## See the maps

For a few photos spanning low → high AQI, we show the RGB image next to its transmission
and inverted-saturation maps. Look for: hazier (higher-AQI) photos tend to be *brighter*
(less dark) in the transmission map, and sky lights up in the inverted-saturation map.

In [ ]:
qs = df["pm25"].quantile([0.1, 0.5, 0.9]).values
picks = [(df["pm25"] - v).abs().idxmin() for v in qs]

fig, axes = plt.subplots(len(picks), 3, figsize=(9, 3 * len(picks)))
for ax_row, idx in zip(axes, picks):
    row = df.loc[idx]
    rgb, t, s = physics.compute_maps(data.get_image(ds, int(row["_row"])), size=cfg["data"]["image_size"])
    ax_row[0].imshow(rgb); ax_row[0].set_title("RGB  (AQI=%.0f)" % row["pm25"])
    ax_row[1].imshow(t, cmap="viridis", vmin=0, vmax=1); ax_row[1].set_title("transmission")
    ax_row[2].imshow(s, cmap="magma", vmin=0, vmax=1); ax_row[2].set_title("inverted saturation")
    for a in ax_row: a.axis("off")
plt.tight_layout(); plt.show()

## Build the physics-map cache (run once)

We precompute both maps for every image and store them to Drive as one memory-mapped
`uint8` array (~1.1 GB). Phase 5 reads from this cache instead of recomputing the Dark
Channel Prior on every epoch.

> ⏳ On the full dataset this takes ~10–15 minutes. It's idempotent — re-running detects
> the existing cache and skips.

In [ ]:
cache_dir = cfg["data"]["cache_dir"]
os.makedirs(cache_dir, exist_ok=True)
cache_path = os.path.join(cache_dir, "physics_maps_%d.npy" % cfg["data"]["image_size"])

if os.path.exists(cache_path):
    cache = physics.load_map_cache(cache_path)
    print("cache already exists:", cache_path, cache.shape)
else:
    physics.build_map_cache(
        len(ds), lambda i: data.get_image(ds, i), cache_path,
        size=cfg["data"]["image_size"], patch=cfg["physics"]["dcp_patch"],
        omega=cfg["physics"]["dcp_omega"], top_frac=cfg["physics"]["atmos_top_frac"],
        t_min=cfg["physics"]["t_min"])
    cache = physics.load_map_cache(cache_path)
    print("built cache:", cache_path, cache.shape)

## Confirm the 5-channel input

We assemble one training tensor to check its shape (5, 224, 224) and channel ranges: the
RGB channels are ImageNet-normalised (roughly centred on 0); the two physics channels
stay in [0, 1].

In [ ]:
r = int(df["_row"].iloc[0])
x = physics.five_channel_cached(data.get_image(ds, r), cache, r, size=cfg["data"]["image_size"])
print("input shape:", x.shape)
print("RGB means:", [round(float(x[c].mean()), 2) for c in range(3)])
print("transmission range: [%.3f, %.3f]" % (x[3].min(), x[3].max()))
print("inv-sat range:      [%.3f, %.3f]" % (x[4].min(), x[4].max()))

## What's next

Every photo can now become a 5-channel tensor quickly (RGB decoded on the fly + maps
from cache). **Next:** `04_model.ipynb` — the EfficientNet-B0 backbone widened to 5
input channels, with three monotone quantile heads.